# Four-method AI text detector — T4 training and inference

Upload the **original competition ZIP** when prompted. Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. No repository checkout or Kaggle credentials are required. Downloads public pretrained weights; supervised fitting uses only the supplied `train.csv`. Nothing is submitted to Kaggle.

The current accepted reference-035 recipe uses weights **0.35 / 0.234 / 0.156 / 0.26** (reference, dual-LM, DistilRoBERTa, DeBERTa), reported at public AUROC **0.97960**. It combines the original lexical/style reference, GPT-2 + GPT-2 Large predictability features, DistilRoBERTa, and DeBERTa-v3-base. Final fitting uses 9,200 supplied examples; 800 separate examples fit fixed quantile calibration. The prediction CSV keeps the test IDs and row order.

Allow roughly 1–2 hours on a T4, plus downloads (large-model training feature extraction alone took 34 minutes in our run). Actual availability and throughput vary. Keep the runtime connected. Download the final bundle before Colab resets. The private competition score remains unknown; local robustness tests do not guarantee a win.


In [ ]:
import os
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
from pathlib import Path
import json, subprocess, sys
PROJECT=Path('/content/four_method_detector')
PROJECT.mkdir(parents=True,exist_ok=True)
FILES={'README.md': '# Human or AI? · Group Number 37\n\n**Kaggle team:** Group Number 37\n\n| Team member | NetID |\n| --- | --- |\n| Alexander Siladie | `aps335` |\n| Cris Nicoglu | `cs2644` |\n| Derrick Cardenas | `djc486` |\n\nOur final submission combines a lexical and style reference, CatBoost on GPT-2 and GPT-2 Large features, DistilRoBERTa, and DeBERTa-v3-base. We normalize each component using quantiles fitted on a separate calibration set and blend them with weights 0.350, 0.234, 0.156, and 0.260. The public Kaggle AUROC was 0.97960.\n\n## Setup\n\nThe full workflow was run on a Colab T4 with Python 3.13.16 and CUDA PyTorch 2.14.1. For Colab, open [Four_Method_Robust_Colab.ipynb](Four_Method_Robust_Colab.ipynb), select a T4 runtime, run all cells, and provide the competition ZIP when prompted. The notebook trains the current recipe and generates `submission_four_methods.csv`. The command-line prediction command below writes `submission.csv`.\n\nFor the command line, use a Python 3.13 environment with CUDA-enabled PyTorch installed. Download the supplied CSVs from Kaggle into `data/` at the repository root. From that root, run:\n\n```bash\ncd submissions/group-number-37\npython -m pip install -r requirements-robust.txt\nexport USE_TF=0 USE_FLAX=0\n```\n\nThe pretrained weights download on first use. Keep data, generated predictions, feature caches, and fitted weights outside Git.\n\n## Evaluate on labeled validation data\n\n```bash\npython train_robust.py --data ../../data --output validation_model --validation --recipe current\npython validate_robust.py --data ../../data --model validation_model\n```\n\nSeed 3737 sets the stratified 8,000/2,000 split and the model seeds. Seeds 9407 and 9413 divide the 8,000 development rows into 6,400 fitting rows, 800 calibration rows, and 800 selection rows. The evaluation script reports AUROC on the fixed 2,000-row audit for original text and formatting stress tests. Its output is `validation_model/audit_report.json`.\n\nThe earlier model comparisons and learning-curve and pseudo-labeling studies are in `experiments/`. Their findings are in [reports/FINDINGS.md](reports/FINDINGS.md). The older `train.py` and `predict.py` reproduce the three-method control. Its separate environment uses Python 3.12 and `requirements.txt`.\n\n## Generate the Kaggle CSV\n\n```bash\npython train_robust.py --data ../../data --output final_model --recipe current\npython predict_robust.py --model final_model --input ../../data/test.csv --output submission.csv --recipe current\n```\n\nFinal fitting uses 9,200 labeled rows and reserves 800 for calibration. Prediction writes the original IDs and AI scores to `submission.csv`. The submitted CSV was generated using the fast reblend from the same fitted components. Full prediction implements the same four-component weights and can also score a new CSV with `id,text` columns. See [TEAM_HANDOFF.md](TEAM_HANDOFF.md) for the reblend workflow.\n\n## Team contributions\n\n- Alex developed the initial notebook and led the final four-component ensemble, expanded LM features, DeBERTa integration, calibration and robustness checks, T4 workflow, and accepted reblend with Codex assistance. His development commits are preserved in this branch.\n- Cris developed the controlled experiments, learning curves, pseudo-labeling studies, dropout fix, domain checks, and earlier three-way blend.\n- Derrick explored poetry-specific ranking methods and tested three Kaggle variants. Those experiments are documented in the report and Kaggle submission descriptions.\n\n## AI assistance\n\nWe used OpenAI Codex for coding, debugging, running experiments, explanations, and drafting and editing the report. Alex directed the final ensemble workflow and robustness comparisons and tested the exported Kaggle submissions. Automated checks covered data splits, prediction IDs and order, score ranges, and agreement between full scoring and the fast reblend. We used the checks and reported results to guide the final recipe. This matches Q6.3 in the report.\n', 'ROBUST_ENSEMBLE.md': "> Historical record of the earlier four-method recipe. Use README.md for the current submission and commands.\n\n> **Current accepted recipe:** use [TEAM_HANDOFF.md](TEAM_HANDOFF.md) and `robust_validation/current_recipe.json`. The default inference weights are now `[0.35, 0.234, 0.156, 0.26]`, reported at public AUROC 0.97960. The report below records the earlier `[0.375, 0.225, 0.15, 0.25]` recipe, its fitted artifact, and its validation results. Historical hashes and checks describe that original artifact, not a new training run.\n\n# Recorded four-method baseline validation\n\nOpen **Four_Method_Robust_Colab.ipynb** in Colab, select a **T4 GPU**, and Run all. Upload the original competition ZIP when prompted. The notebook is self-contained and needs no repository checkout. Save its prediction CSV and fitted model bundle before the runtime expires. Nothing is submitted to Kaggle.\n\nThis adds a tested candidate alongside the committed pipeline. It does not replace the team's main recipe or establish a better Kaggle score. The generating code for the team's latest F submissions was not available in the reviewed snapshot, so the control here is the committed three-way pipeline, refitted on identical training rows.\n\n## Fixed recipe\n\n| Method | Weight | What it contributes |\n|---|---:|---|\n| Original lexical/style reference | 0.375 | Word/character TF-IDF logistic regression plus style CatBoost; preserves the team's successful reference |\n| Dual-language-model CatBoost | 0.225 | 15 predictability features each from GPT-2 and GPT-2 Large, plus their 15 differences |\n| DistilRoBERTa | 0.150 | Existing fine-tuned semantic classifier |\n| DeBERTa-v3-base | 0.250 | A different fine-tuned architecture, with head/tail passage coverage |\n\nBoth neural classifiers use two epochs and cleaned text. DeBERTa freezes vocabulary embeddings and uses gradient accumulation in full precision. GPT-2 Large feature extraction ran successfully on a T4 in full precision: 10,000 passages in 33.7 minutes, peak allocated GPU memory 6.32 GiB.\n\nComponent scales use quantiles fitted on 800 held-out **training** rows, pooling original, flattened, and short-line views. They are frozen at inference: a new file does not redefine ranks or ensemble weights. Quantiles are scores, not calibrated authorship probabilities.\n\nFinal fitting uses 9,200 supplied labeled rows, reserving 800 for calibration. No external labeled text, pseudo-labels, or competition test labels are used. Pretrained general-purpose model weights are downloaded publicly. The supplied unlabeled CSV is unused.\n\n## Local evidence\n\n| Frozen global audit, 2,000 rows | Committed recipe control | Candidate |\n|---|---:|---:|\n| Original text | 0.999212 | **0.999603** |\n| Flattened whitespace | 0.980368 | **0.994209** |\n| Synthetic short lines | 0.987352 | **0.996970** |\n\nBoth models fit the same 6,400 rows; 800 separate rows fit calibration and 800 select the recipe. The recipe was selected across the selection split plus two content-held-out development groups (conversational text and recipes). Each component was fitted without its held-out content group. Candidates had to retain the control on all nine development scenarios. Among eligible candidates, selection maximized mean native content-held-out AUC, then worst formatting AUC. The chosen recipe used 45 dual-LM features and the fixed weights above.\n\nOriginal content-held-out development AUROC increased from 0.996467 to 0.997975 for conversational text and from 0.994665 to 0.997803 for recipes. These are **selection results**, not untouched test estimates. See `robust_validation/recipe.json` for every scenario, split hash, and the 243 searched combinations.\n\nA stratified paired bootstrap of the fixed global audit (1,000 replicates) gives an original-text AUC difference of +0.000391, conditional 95% interval [+0.000016, +0.000765]. Formatting gains have wider absolute improvements. These intervals do not account for prior experimentation, source dependence, or distribution shift. This audit seed has been used in the project before; it is not a pristine independent benchmark. Do not keep tuning against these audit results.\n\nThe production refactor reproduces all three cached audit AUCs, with maximum score differences below 2.4e-9 on the local check. Normalization changes preserve all 10,000 original cleaned inputs and make all whitespace stress views identical for neural tokenization, including a literal HTML line-break edge case.\n\nA further news-like group was evaluated after freezing the recipe: 4,830 fitting rows, 800 calibration rows, 770 held-out rows (377 human, 393 AI). Native AUC rose from 0.994465 to **0.996504**; flattened AUC from 0.952903 to 0.979597; short-line AUC from 0.930056 to 0.983828. No weights were changed after this check. See `robust_validation/frozen_news_check.json`. It is another within-dataset transfer check, not an external benchmark.\n\nSynthetic wrapping tests do not prove performance on real human poetry or unknown generators. A high local AUC cannot guarantee private leaderboard performance, and the model has not been submitted to Kaggle. Preserve the existing pipeline as a comparison until the team evaluates the candidate.\n\n## Command line\n\nUse Python 3.11+ and an existing CUDA PyTorch installation (tested locally with 2.6 and on Colab with 2.14.1+cu130). Install the additional dependencies:\n\n```bash\npython -m pip install -r requirements-robust.txt\nUSE_TF=0 USE_FLAX=0 python train_robust.py --data data --output final_model\nUSE_TF=0 USE_FLAX=0 python check_robust.py --model final_model --input data/test.csv --recipe baseline\nUSE_TF=0 USE_FLAX=0 python predict_robust.py --model final_model --input data/test.csv --output submission_four_methods.csv --recipe baseline\n```\n\n`data/` should contain the original train/test CSVs. Fresh training takes approximately 1–2 hours on T4 including feature extraction and prediction, plus downloads. CPU/Mac device fallback is available, but full training needs sufficient RAM and can be substantially slower. Use the tested T4 route for this project; the full recipe has not been verified to fit an 8 GB Mac. Existing feature caches can be supplied with `--small-features` and `--large-features`; dataset digest, model metadata, and matrix shape are checked. Completed encoder checkpoints resume only within the same source/configuration manifest. Changing code requires a fresh output folder.\n\nTo reproduce the frozen global audit without training on its labels:\n\n```bash\nUSE_TF=0 USE_FLAX=0 python train_robust.py --data data --output audit_model --validation\nUSE_TF=0 USE_FLAX=0 python validate_robust.py --data data --model audit_model\n```\n\nThe additional frozen news check can be reproduced with `validate_content_robust.py --data data --features audit_model/small_features.npz --large-features audit_model/large_features.npz --recipe robust_validation/recipe.json --output news_check`. It retrains both neural classifiers, excluding that content cluster, and takes additional GPU time. Cached features contain no fitted authorship labels.\n\nDo not audit a final model trained on 9,200 rows; that includes the old audit labels. Fine-tuning can vary numerically between GPU/software versions, so exact score equality is not promised for independent retraining.\n\n## Professor's later evaluation\n\nKeep the fitted bundle intact, including both fine-tuned neural folders, `model.joblib`, and source. Then run `predict_robust.py --model final_model --input new_test.csv --output predictions.csv`. Input needs unique `id` values and nonempty `text`; output contains `id,label` in the same order. No training, threshold selection, or recalibration is performed on the new file. GPU is recommended.\n\nPublic pretrained GPT-2 weights download automatically. For disconnected inference, cache the original model/tokenizer files under `gpt2/` and `gpt2-large/`, then pass `--offline-root`. GPT-2 Large is pinned to revision `32b71b12589c2f8d625668d2335a01cac3249519`. The fitted bundle is about 0.8 GiB before optional offline pretrained weights; share it in team storage rather than committing it to GitHub. Load only trusted joblib bundles. The notebook uses `package_robust.py` to verify the export includes model/tokenizer files and package metadata, while excluding data and feature caches.\n\n## Completed final run\n\nThe final 9,200-row fit and separate 800-row calibration completed on Tesla T4. Both neural classifiers trained for two epochs. The fit took **33.46 minutes** using the already completed language-feature caches, with **4.62 GiB** peak allocated GPU memory during fitting. This timing excludes initial feature extraction, final checks, test inference, and archive creation.\n\nThe fitted final detector passed the full inference batch-size and row-order check: maximum score difference **4.21e-7** on eight passages. It then generated all **3,000** test predictions. Independent local verification confirmed the original test IDs and sample row order, finite scores in [0,1], and the downloaded CSV checksum. The model ZIP also passed SHA-256 and ZIP CRC checks; its serialized classifier, both fine-tuned checkpoints/tokenizers, 45-feature LM classifier, and fixed weights were verified locally.\n\nThe ZIP contains 49 files and is 860,134,885 bytes (0.80 GiB), including code and fitted models. Competition CSVs and feature caches are excluded. Its SHA-256 is `4bd236a9432666055408a15307de62a296adc867876d03a926d1a5f138c1dce8`. See `final_training_complete.json`, `final_inference_check.json`, and `local_verification.json` under `robust_validation/` for provenance.\n\nThe AUC results above belong to the separate validation fits. The final 9,200-row model has no measured Kaggle/private-test score. Prediction-file integrity checks do not measure classification accuracy.\n", 'START_HERE.md': '# Start here\n\nFollow the [course README](README.md) to set up the environment, run validation, and generate the Kaggle CSV.\n', 'TEAM_HANDOFF.md': "# Current team handoff\n\nStart with the [course README](README.md) for setup and reproduction commands.\n\nThe current accepted submission is **reference-035**, reported by Alex at **0.97960** on the public leaderboard. The earlier four-method submission scored 0.97956. The private score is unknown.\n\n## Current recipe\n\n| Component | Weight |\n|---|---:|\n| Original word/character/style reference | 0.350 |\n| GPT-2 + GPT-2 Large feature classifier | 0.234 |\n| DistilRoBERTa | 0.156 |\n| DeBERTa-v3-base | 0.260 |\n\nThe models and quantile calibration are shared with the earlier submission; only the final weights changed. Base fitting uses 9,200 supplied labeled examples, with 800 separate examples for calibration. No extra labeled datasets are used. Scores are for ranking, not literal authorship probabilities.\n\n`predict_robust.py` now defaults to `--recipe current`, including with the earlier saved model bundle. `--recipe baseline` reproduces the earlier 0.97956 recipe; `--recipe saved` explicitly uses the weights serialized inside a bundle. New final training saves the current weights. `train_robust.py --validation` defaults to the baseline recipe to reproduce the historical audit; pass `--recipe current` to evaluate the accepted reblend instead.\n\n## Files to start with\n\n- `Four_Method_Robust_Colab.ipynb`: self-contained fresh T4 training, prediction, checks, and model export; upload the original competition ZIP.\n- `train_robust.py`, `predict_robust.py`, `reblend_robust.py`: training, full inference, and a fast CPU-only reblend.\n- `robust_validation/current_recipe.json`: current weights, paired checks, and reported public scores.\n- `ROBUST_ENSEMBLE.md`: the earlier four-method recipe's detailed validation record and timing/provenance. Its historical 0.375 reference weight is not the current default.\n\nThe older `train.py` and `predict.py` remain the team's original three-way control. Use the `*_robust.py` commands for this pipeline.\n\n## Reuse a trained bundle\n\nInstall `requirements-robust.txt` alongside a suitable PyTorch installation (CUDA/T4 recommended), extract the existing trained bundle, and run:\n\n```bash\nUSE_TF=0 USE_FLAX=0 python predict_robust.py --model final_model --input data/test.csv --output submission_current.csv --recipe current\n```\n\nThis also scores a professor-provided later CSV with `id,text`, preserving its row order. No new fitting or recalibration occurs during inference. Public pretrained GPT-2 weights download automatically; offline copies can be supplied with `--offline-root`.\n\nIf you already have the SAME fitted model's baseline 0.375/0.225/0.15/0.25 predictions for that input file, the shortcut avoids repeating all neural inference:\n\n```bash\nUSE_TF=0 USE_FLAX=0 python reblend_robust.py --model four_method_bundle.zip --input data/test.csv --baseline submission_four_methods.csv --output submission_current.csv\n```\n\nThe shortcut computes only the reference score and reconstructs the combined contribution of the other components from the baseline CSV. It requires the same model, input IDs/order, and baseline recipe. Do not feed it the already-reblended 0.97960 CSV. It preserves the baseline file.\n\n## Reproduce or improve training\n\nIn Colab: upload the notebook, select T4 GPU, Run all, and upload the original competition ZIP. Expect about 1.5–2 hours for a fresh complete run, plus downloads; the recorded final fit alone took 33.46 minutes with language features already cached. The notebook produces the CURRENT recipe by default and never submits anything.\n\nTraining features and fitted checkpoints are reusable. Keep them outside Git. The trainer supports `--small-features` and `--large-features` with matching dataset/model metadata. Source/configuration changes require a fresh output directory; reuse compatible feature files explicitly instead of recomputing them. Model ZIPs, feature caches, and original data are retained locally; this source handoff does not provide a hosted weights download. Obtain the existing fitted bundle from Alex or reproduce it with the notebook.\n\n## Evidence and next improvements\n\nThe accepted reblend's original-text audit AUC is 0.999558 (earlier recipe: 0.999603), flattened AUC 0.994925 (0.994209), and short-line AUC 0.997057 (0.996970). All three paired news checks also improved. These are local validation fits, not private-test estimates; the public scores above were reported by Alex.\n\nKnown risks: the audit split has prior project exposure; content-development checks influenced selection; human poetry is poorly covered by supplied training labels. Synthetic wrapping does not establish real human-poetry performance. Keep data groups/near-duplicates together during evaluation, fit preprocessing only on each fitting partition, and separate model selection from evaluation. Use the supplied labels only; compare any new detector against this baseline on content and formatting checks before replacing it.\n\nThe 800-row calibration partition is excluded from supervised component fitting. It fits fixed component quantiles; inference never ranks against the other rows of the input file. Current weights are fixed for future data. The submitted CSV already includes all 3,000 test IDs; Kaggle's quoted 30/70 split changes which rows contribute to public/private scoring, not which predictions are in this file.\n", 'check_robust.py': '"""Verify complete inference on different batch sizes and input order."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom predict_robust import predict_rows\nfrom src.recipes import recipe_weights\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--model\',type=Path,required=True)\n    p.add_argument(\'--input\',type=Path,required=True)\n    p.add_argument(\'--recipe\',choices=(\'current\',\'baseline\',\'saved\'),default=\'current\');a=p.parse_args()\n    texts=pd.read_csv(a.input,keep_default_na=False).text.head(8).tolist()\n    bundle=joblib.load(a.model/\'model.joblib\')\n    singular=predict_rows(bundle,a.model,texts,batch=1,recipe=a.recipe)\n    together=predict_rows(bundle,a.model,texts,batch=8,recipe=a.recipe)\n    reversed_scores=predict_rows(bundle,a.model,texts[::-1],batch=8,recipe=a.recipe)[::-1]\n    difference=max(float(np.max(np.abs(singular-together))),float(np.max(np.abs(together-reversed_scores))))\n    if not np.allclose(singular,together,atol=1e-4,rtol=1e-4) or not np.allclose(together,reversed_scores,atol=1e-4,rtol=1e-4):\n        raise AssertionError(f\'Inference depends on batch/order beyond numerical tolerance: {difference}\')\n    result={\'passed\':True,\'rows\':len(texts),\'maximum_score_difference\':difference,\n            \'recipe\':a.recipe,\'weights\':recipe_weights(bundle,a.recipe).tolist(),\n            \'model_sha256\':hashlib.sha256((a.model/\'model.joblib\').read_bytes()).hexdigest()}\n    (a.model/\'inference_check.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n    print(json.dumps(result,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'package_robust.py': '"""Export trusted fitted models and inference code without competition data/caches."""\nimport argparse,json,zipfile\nfrom pathlib import Path\n\ndef package(directory,model,output):\n    required=(\'model.joblib\',\'training_complete.json\',\'inference_check.json\')\n    for name in required:\n        if not (model/name).is_file():raise ValueError(f\'Model bundle is incomplete: {name}\')\n    if not json.loads((model/\'inference_check.json\').read_text()).get(\'passed\'):raise ValueError(\'Inference check must pass before export.\')\n    if output.resolve().is_relative_to(model.resolve()):raise ValueError(\'Write the ZIP outside the model folder.\')\n    with zipfile.ZipFile(output,\'w\',zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(directory.rglob(\'*\')):\n            if not path.is_file() or \'__pycache__\' in path.parts or path==output:continue\n            relative=path.relative_to(directory)\n            is_source=(path.parent==directory and path.suffix in (\'.py\',\'.txt\',\'.md\')) or (relative.parts[0]==\'src\' and path.suffix==\'.py\')\n            is_evidence=relative.parts[0]==\'robust_validation\' and path.suffix==\'.json\'\n            is_model=model in path.parents and path.suffix in (\'.joblib\',\'.json\',\'.safetensors\',\'.bin\',\'.txt\',\'.model\')\n            if path.name.startswith(\'_\') and path.name!=\'__init__.py\':continue\n            if is_source or is_evidence or is_model:archive.write(path,relative)\n    with zipfile.ZipFile(output) as archive:\n        names=archive.namelist()\n        for name in (\'src/__init__.py\',\'src/robust_ensemble.py\',\'predict_robust.py\',str(model.relative_to(directory)/\'model.joblib\')):\n            if name not in names:raise AssertionError(f\'Export omitted {name}\')\n    return {\'bundle\':str(output),\'bytes\':output.stat().st_size,\'files\':len(names)}\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--model\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args();print(json.dumps(package(Path(__file__).resolve().parent,a.model.resolve(),a.output.resolve()),indent=2))\n\nif __name__==\'__main__\':main()\n', 'predict_robust.py': '"""Score any new id,text CSV with fixed models; never submit to Kaggle."""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom transformers import AutoModelForSequenceClassification,AutoTokenizer\nfrom src.encoder import DEVICE,predict as legacy_predict\nfrom src.encoder_v2 import predict_encoder\nfrom src.lm_features import MODELS,extract,combine\nfrom src.recipes import recipe_weights\n\n\ndef predict_rows(bundle,directory,texts,offline_root=None,batch=8,recipe=\'current\'):\n    if bundle.get(\'lm_models\')!=list(MODELS):raise ValueError(\'Expected the validated dual-LM model bundle.\')\n    features=combine(*(extract(texts,model_id,batch,offline_root) for model_id in MODELS))\n    deberta=predict_encoder(texts,directory/bundle[\'deberta_dir\'],batch_size=batch)\n    distil_dir=directory/bundle[\'distil_dir\']\n    model=AutoModelForSequenceClassification.from_pretrained(distil_dir,token=False,attn_implementation=\'eager\').to(DEVICE)\n    tok=AutoTokenizer.from_pretrained(distil_dir,token=False)\n    distil=legacy_predict(model,tok,list(texts),batch=batch)\n    del model;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    ensemble=bundle[\'ensemble\']\n    components=ensemble.components(texts,features,distil,deberta)\n    result=np.clip(ensemble.calibration.transform(components)@recipe_weights(bundle,recipe),0,1)\n    if not np.isfinite(result).all() or np.any((result<0)|(result>1)):raise ValueError(\'Invalid scores.\')\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--model\',type=Path,required=True)\n    p.add_argument(\'--input\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--offline-root\',type=Path,help=\'Optional directory containing gpt2 and gpt2-large pretrained folders\')\n    p.add_argument(\'--batch\',type=int,default=8)\n    p.add_argument(\'--recipe\',choices=(\'current\',\'baseline\',\'saved\'),default=\'current\',\n                   help=\'current: reported 0.97960 reblend; baseline: reported 0.97956; saved: bundle weights\')\n    a=p.parse_args()\n    if a.batch<1:raise ValueError(\'Batch size must be positive.\')\n    frame=pd.read_csv(a.input,keep_default_na=False)\n    if not {\'id\',\'text\'}.issubset(frame.columns) or not len(frame):raise ValueError(\'Expected a nonempty CSV with id and text columns.\')\n    if frame.id.duplicated().any():raise ValueError(\'Input IDs must be unique.\')\n    if not frame.text.map(lambda x:isinstance(x,str) and bool(x.strip())).all():raise ValueError(\'Every passage must contain text.\')\n    bundle=joblib.load(a.model/\'model.joblib\')\n    result=predict_rows(bundle,a.model,frame.text.tolist(),a.offline_root,a.batch,a.recipe)\n    a.output.parent.mkdir(parents=True,exist_ok=True)\n    pd.DataFrame({\'id\':frame.id,\'label\':result}).to_csv(a.output,index=False)\n    print(json.dumps({\'rows\':len(frame),\'output\':str(a.output),\'recipe\':a.recipe,\'weights\':recipe_weights(bundle,a.recipe).tolist(),\n                      \'fit_rows\':bundle.get(\'fit_rows\'),\'stage\':bundle.get(\'stage\')}),flush=True)\n\nif __name__==\'__main__\':main()\n', 'predictions/README.md': '# Current accepted predictions\n\n`submission_current_best.csv` is the unchanged 3,000-row reference-035 prediction file reported by Alex at public AUROC **0.97960**. It contains only `id,label`, with no source text or ground-truth test labels. The private score is unknown.\n\nSHA-256: `151187349693353ff92d3bb8b3c5f5a12ecfef03640f2f78efb3afffe816b4d6`.\n', 'reblend_robust.py': '"""Produce the accepted reblend from the same model\'s baseline CSV, without GPU inference."""\nimport argparse\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import hstack\n\nfrom src.baseline import style_matrix\nfrom src.calibration import RankCalibration\nfrom src.recipes import BASELINE_WEIGHTS, CURRENT_WEIGHTS\n\n\ndef load_bundle(path):\n    if path.is_file():\n        with zipfile.ZipFile(path) as archive:\n            return joblib.load(io.BytesIO(archive.read(\'final_model/model.joblib\')))\n    return joblib.load(path/\'model.joblib\')\n\n\ndef calibrated_reference(bundle, texts):\n    ensemble=bundle[\'ensemble\']; model=ensemble.classical\n    if not isinstance(ensemble.calibration,RankCalibration):\n        raise ValueError(\'This shortcut requires the accepted independent quantile calibration.\')\n    x=hstack((model.word.transform(texts),model.char.transform(texts)),format=\'csr\')\n    raw=.35*model.lexical.decision_function(x)+.65*model.style.predict(style_matrix(texts),prediction_type=\'RawFormulaVal\')\n    columns=np.zeros((len(texts),4));columns[:,0]=raw\n    return ensemble.calibration.transform(columns)[:,0]\n\n\ndef reblend_scores(baseline, reference):\n    baseline=np.asarray(baseline,dtype=float);reference=np.asarray(reference,dtype=float)\n    if baseline.shape!=reference.shape or baseline.ndim!=1 or not np.isfinite(baseline).all() or not np.isfinite(reference).all():\n        raise ValueError(\'Expected matching finite score vectors.\')\n    # All three other components increase by the same factor in the accepted recipe.\n    ratios=CURRENT_WEIGHTS[1:]/BASELINE_WEIGHTS[1:]\n    if not np.allclose(ratios,ratios[0]):raise ValueError(\'This shortcut is invalid for the selected weights.\')\n    rest=(baseline-BASELINE_WEIGHTS[0]*reference)/(1-BASELINE_WEIGHTS[0])\n    if np.any((rest < -1e-6)|(rest > 1+1e-6)):\n        raise ValueError(\'Baseline scores are inconsistent with this model/recipe. Regenerate with --recipe baseline.\')\n    return np.clip(CURRENT_WEIGHTS[0]*reference+(1-CURRENT_WEIGHTS[0])*rest,0,1)\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--model\',type=Path,required=True,help=\'Extracted model directory or trusted four_method_bundle.zip\')\n    p.add_argument(\'--input\',type=Path,required=True,help=\'The same id,text CSV used for the baseline predictions\')\n    p.add_argument(\'--baseline\',type=Path,required=True,help=\'CSV generated by the SAME fitted model with --recipe baseline\')\n    p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args()\n    if a.output.resolve()==a.baseline.resolve():raise ValueError(\'Preserve the baseline; choose a different output file.\')\n    text=pd.read_csv(a.input,keep_default_na=False);baseline=pd.read_csv(a.baseline,keep_default_na=False)\n    if not {\'id\',\'text\'}.issubset(text.columns) or list(baseline.columns)!=[\'id\',\'label\'] or not len(text):\n        raise ValueError(\'Expected nonempty id,text input and id,label baseline.\')\n    if not text.id.equals(baseline.id) or text.id.duplicated().any():raise ValueError(\'IDs and row order must match exactly and be unique.\')\n    if not text.text.map(lambda value:isinstance(value,str) and bool(value.strip())).all():raise ValueError(\'Every passage must contain text.\')\n    scores=baseline.label.to_numpy(dtype=float)\n    if not np.isfinite(scores).all() or np.any((scores<0)|(scores>1)):raise ValueError(\'Invalid baseline scores.\')\n    reference=calibrated_reference(load_bundle(a.model),text.text.to_numpy())\n    result=reblend_scores(scores,reference)\n    a.output.parent.mkdir(parents=True,exist_ok=True)\n    pd.DataFrame({\'id\':text.id,\'label\':result}).to_csv(a.output,index=False)\n    print(json.dumps({\'rows\':len(text),\'output\':str(a.output),\'recipe\':\'current\',\'weights\':CURRENT_WEIGHTS.tolist(),\n                      \'baseline_sha256\':hashlib.sha256(a.baseline.read_bytes()).hexdigest(),\n                      \'output_sha256\':hashlib.sha256(a.output.read_bytes()).hexdigest()}))\n\n\nif __name__==\'__main__\':main()\n', 'requirements-robust.txt': '# Colab supplies CUDA-enabled torch. Tested with torch 2.6 and newer.\ntransformers==4.51.3\naccelerate==1.7.0\nscikit-learn==1.6.1\ncatboost==1.2.10\nsentencepiece==0.2.2\nprotobuf>=5,<7\npandas>=2.2,<3\nnumpy>=1.26,<3\nscipy>=1.13,<2\njoblib>=1.4,<2\nsafetensors>=0.4\n', 'robust_validation/audit_uncertainty.json': '{\n  "original": {\n    "auc_difference": 0.0003910000000000302,\n    "paired_bootstrap_95_percent_interval": [\n      1.5974999999909813e-05,\n      0.0007650250000000219\n    ]\n  },\n  "flat": {\n    "auc_difference": 0.013840999999999881,\n    "paired_bootstrap_95_percent_interval": [\n      0.010746350000000066,\n      0.01695423750000024\n    ]\n  },\n  "short_lines": {\n    "auc_difference": 0.009617999999999904,\n    "paired_bootstrap_95_percent_interval": [\n      0.00678453750000004,\n      0.012950775000000086\n    ]\n  }\n}\n', 'robust_validation/colab_parity_check.json': '{\n  "original": {\n    "auc": 0.999603,\n    "maximum_refactor_difference": 0.0\n  },\n  "flat": {\n    "auc": 0.994209,\n    "maximum_refactor_difference": 0.0\n  },\n  "short_lines": {\n    "auc": 0.99697,\n    "maximum_refactor_difference": 0.0\n  }\n}\n', 'robust_validation/content_driver_reproduction.json': '{\n  "weights_locked_before_testing": [\n    0.375,\n    0.225,\n    0.15,\n    0.25\n  ],\n  "score_mode": "quantile",\n  "lm_mode": "both",\n  "topics": [\n    {\n      "topic": "news",\n      "terms": [\n        "financial",\n        "bbc",\n        "country",\n        "added",\n        "told",\n        "market",\n        "years",\n        "uk",\n        "government",\n        "world",\n        "new",\n        "mr",\n        "company",\n        "year",\n        "said"\n      ],\n      "fit_rows": 4830,\n      "calibration_rows": 800,\n      "held_rows": 770,\n      "held_human": 377,\n      "held_ai": 393,\n      "candidate": {\n        "original": 0.9965375503675056,\n        "flat": 0.9800622296015821,\n        "short_lines": 0.9826675035940632\n      },\n      "control": {\n        "original": 0.9943169930008572,\n        "flat": 0.952224269544617,\n        "short_lines": 0.927035454674307\n      }\n    }\n  ],\n  "pooled_candidate": {\n    "original": 0.9965375503675056,\n    "flat": 0.9800622296015821,\n    "short_lines": 0.9826675035940632\n  },\n  "pooled_control": {\n    "original": 0.9943169930008572,\n    "flat": 0.952224269544617,\n    "short_lines": 0.927035454674307\n  },\n  "scope": "Additional content group with frozen recipe; from the original fitting partition only, not a human-poetry benchmark.",\n  "verification_note": "Production content-check driver; local CPU refit of classical components with the original Colab neural score cache. Platform-dependent classical scores differ slightly; candidate retains gains on every view. Not an independent test."\n}\n', 'robust_validation/current_recipe.json': '{\n  "name": "reference-035",\n  "component_order": [\n    "reference",\n    "dual_lm",\n    "distilroberta",\n    "deberta"\n  ],\n  "weights": [\n    0.35,\n    0.234,\n    0.156,\n    0.26\n  ],\n  "baseline_weights": [\n    0.375,\n    0.225,\n    0.15,\n    0.25\n  ],\n  "score_mode": "quantile",\n  "lm_mode": "both",\n  "lm_models": [\n    "gpt2",\n    "openai-community/gpt2-large"\n  ],\n  "training_sha256": "325e345faf88fb1c4203278098c282e40319a87b4bf17e094d2f235f51d0c07e",\n  "public_score_reported_by_user": 0.9796,\n  "baseline_public_score_reported_by_user": 0.97956,\n  "private_score_known": false,\n  "neural_retraining_required_for_reblend": false,\n  "paired_validation": {\n    "audit": {\n      "original": {\n        "baseline": 0.999603,\n        "candidate": 0.999558\n      },\n      "flat": {\n        "baseline": 0.994209,\n        "candidate": 0.994925\n      },\n      "short_lines": {\n        "baseline": 0.99697,\n        "candidate": 0.997057\n      }\n    },\n    "news_local_reproduction": {\n      "original": {\n        "baseline": 0.9965375503675056,\n        "candidate": 0.996686037486248\n      },\n      "flat": {\n        "baseline": 0.9800622296015821,\n        "candidate": 0.981054393531361\n      },\n      "short_lines": {\n        "baseline": 0.9826675035940632,\n        "candidate": 0.9831669602661969\n      }\n    }\n  },\n  "tradeoff": "Original audit AUC decreased from 0.999603 to 0.999558; formatting audit and all paired news checks improved. Not a guaranteed private-score improvement."\n}\n', 'robust_validation/environment.json': '{\n  "torch": "2.14.1+cu130",\n  "transformers": "4.51.3",\n  "accelerate": "1.7.0",\n  "scikit-learn": "1.6.1",\n  "catboost": "1.2.10",\n  "pandas": "2.2.3",\n  "numpy": "2.4.6",\n  "sentencepiece": "0.2.2",\n  "protobuf": "6.33.6"\n}', 'robust_validation/final_inference_check.json': '{\n  "passed": true,\n  "rows": 8,\n  "maximum_score_difference": 4.212665558611661e-07,\n  "model_sha256": "704879462f00a1837e28bba57e8180fb24c54538a333888a167d5507ba5b0b88"\n}\n', 'robust_validation/final_training_complete.json': '{\n  "training_sha256": "325e345faf88fb1c4203278098c282e40319a87b4bf17e094d2f235f51d0c07e",\n  "source_sha256": "a2a52b6d243409e0630783fb3d0518f13a8dc3486b06ff92c0f8002bf62b06e5",\n  "weights": [\n    0.375,\n    0.225,\n    0.15,\n    0.25\n  ],\n  "fit_rows": 9200,\n  "calibration_rows": 800,\n  "encoder_epochs": 2,\n  "recipe": "four-method-robustness-v1",\n  "stage": "final",\n  "elapsed_minutes": 33.45975911918334,\n  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",\n  "torch": "2.14.1+cu130",\n  "device": "Tesla T4",\n  "peak_gpu_gib": 4.62431526184082\n}\n', 'robust_validation/frozen_news_check.json': '{\n  "weights_locked_before_testing": [\n    0.375,\n    0.225,\n    0.15,\n    0.25\n  ],\n  "score_mode": "quantile",\n  "lm_mode": "both",\n  "topics": [\n    {\n      "topic": "news",\n      "terms": [\n        "financial",\n        "bbc",\n        "country",\n        "added",\n        "told",\n        "market",\n        "years",\n        "uk",\n        "government",\n        "world",\n        "new",\n        "mr",\n        "company",\n        "year",\n        "said"\n      ],\n      "fit_rows": 4830,\n      "calibration_rows": 800,\n      "held_rows": 770,\n      "held_human": 377,\n      "held_ai": 393,\n      "candidate": {\n        "original": 0.9965038032950642,\n        "flat": 0.9795965200018898,\n        "short_lines": 0.9838284028860497\n      },\n      "control": {\n        "original": 0.9944654801195997,\n        "flat": 0.9529025857006904,\n        "short_lines": 0.9300558176578182\n      }\n    }\n  ],\n  "pooled_candidate": {\n    "original": 0.9965038032950642,\n    "flat": 0.9795965200018898,\n    "short_lines": 0.9838284028860497\n  },\n  "pooled_control": {\n    "original": 0.9944654801195997,\n    "flat": 0.9529025857006904,\n    "short_lines": 0.9300558176578182\n  },\n  "scope": "Additional content group with frozen recipe; from the original fitting partition only, not a human-poetry benchmark."\n}\n', 'robust_validation/local_verification.json': '{\n  "passed": true,\n  "archive_crc_passed": true,\n  "bundle_sha256": "4bd236a9432666055408a15307de62a296adc867876d03a926d1a5f138c1dce8",\n  "model_sha256": "704879462f00a1837e28bba57e8180fb24c54538a333888a167d5507ba5b0b88",\n  "bundle_files": 49,\n  "bundle_bytes": 860134885,\n  "dataset_files_in_bundle": false,\n  "lm_classifier_features": 45,\n  "fit_rows": 9200,\n  "calibration_rows": 800,\n  "prediction_rows": 3000\n}\n', 'robust_validation/normalization_check.json': '{\n  "changed_original_inputs": 0,\n  "old_flat_mismatch": 0,\n  "old_short_line_mismatch": 1,\n  "new_flat_mismatch": 0,\n  "new_short_line_mismatch": 0\n}\n', 'robust_validation/parity_check.json': '{\n  "original": {\n    "auc": 0.999603,\n    "maximum_refactor_difference": 2.2895505491504764e-09\n  },\n  "flat": {\n    "auc": 0.994209,\n    "maximum_refactor_difference": 2.354123840309086e-09\n  },\n  "short_lines": {\n    "auc": 0.99697,\n    "maximum_refactor_difference": 2.367612134124286e-09\n  }\n}\n', 'robust_validation/recipe.json': '{\n  "score_mode": "quantile",\n  "lm_mode": "both",\n  "weights": [\n    0.375,\n    0.225,\n    0.15,\n    0.25\n  ],\n  "lm_feature_layout": "small, large, large-minus-small",\n  "lm_models": [\n    "gpt2",\n    "openai-community/gpt2-large"\n  ],\n  "development": {\n    "selection/original": 0.9996,\n    "selection/flat": 0.99363125,\n    "selection/short_lines": 0.99555625,\n    "conversational/original": 0.9979745245180281,\n    "conversational/flat": 0.9906049345325942,\n    "conversational/short_lines": 0.9947520314846062,\n    "recipes/original": 0.9978032524660091,\n    "recipes/flat": 0.993822980538523,\n    "recipes/short_lines": 0.9927805918421754\n  },\n  "development_control": {\n    "selection/original": 0.99918125,\n    "selection/flat": 0.978209375,\n    "selection/short_lines": 0.98476875,\n    "conversational/original": 0.9964666349384906,\n    "conversational/flat": 0.9671685487243831,\n    "conversational/short_lines": 0.9733579467062788,\n    "recipes/original": 0.9946654225539856,\n    "recipes/flat": 0.9581111703545722,\n    "recipes/short_lines": 0.9621007731271661\n  },\n  "audit": {\n    "original": 0.999603,\n    "flat": 0.994209,\n    "short_lines": 0.99697\n  },\n  "control_audit": {\n    "original": 0.999212,\n    "flat": 0.980368,\n    "short_lines": 0.987352\n  },\n  "audit_passed": true,\n  "training_sha256": "325e345faf88fb1c4203278098c282e40319a87b4bf17e094d2f235f51d0c07e",\n  "selection_rule": "No regression on any development scenario; maximize mean native content-holdout AUC, then worst formatting AUC.",\n  "eligible_candidates": 80,\n  "searched_candidates": 243\n}\n', 'robust_validation/validation_inference_check.json': '{\n  "passed": true,\n  "rows": 8,\n  "maximum_score_difference": 1.4282454618985696e-07,\n  "model_sha256": "17ae505048d98a4b62b55a095d5f9647bb39078ad864536aed6af74eaff9af04"\n}\n', 'src/__init__.py': '', 'src/attacks.py': '"""label-preserving text edits. used to stress test models and as training augmentation.\n\nthese are written from scratch from general knowledge of how detectors get evaded. no external\nlabeled data is involved, only edits applied to the passages we already have labels for.\n"""\nimport random\nimport re\n\nHOMOGLYPHS = {"a": "а", "e": "е", "o": "о", "c": "с", "p": "р", "i": "і", "x": "х", "y": "у"}\nZERO_WIDTH = "\u200b"\nSPELLING = {"color": "colour", "center": "centre", "organize": "organise", "realize": "realise", "favorite": "favourite",\n            "behavior": "behaviour", "analyze": "analyse", "theater": "theatre", "defense": "defence", "labor": "labour"}\nTYPOS = {"the": "teh", "and": "adn", "receive": "recieve", "until": "untill", "their": "thier", "which": "wich",\n         "because": "becuase", "people": "poeple", "definitely": "definately", "patience": "patients"}\nSYNONYMS = {"important": "crucial", "big": "large", "help": "assist", "use": "employ", "show": "demonstrate", "many": "numerous",\n            "start": "begin", "buy": "purchase", "end": "finish", "good": "fine", "hard": "difficult", "often": "frequently",\n            "however": "nevertheless", "also": "additionally", "get": "obtain", "need": "require", "small": "little",\n            "shows": "demonstrates", "uses": "employs", "helps": "assists"}\n\n\ndef _swap_chars(t, table, rate, r):\n    return "".join(table[c] if c in table and r.random() < rate else c for c in t)\n\n\ndef _swap_words(t, table, rate, r):\n    def sub(m):\n        w = m.group(0)\n        n = table.get(w.lower())\n        if n is None or r.random() > rate:\n            return w\n        return n.capitalize() if w[0].isupper() else n\n    return re.sub(r"[A-Za-z]+", sub, t)\n\n\ndef flatten_ws(t, r):\n    return re.sub(r"\\s+", " ", t).strip()\n\n\ndef drop_blank_lines(t, r):\n    return re.sub(r"\\n\\s*\\n", "\\n", t)\n\n\ndef add_blank_lines(t, r):\n    return t.replace(". ", ".\\n\\n") if r.random() < .5 else t.replace("\\n", "\\n\\n")\n\n\ndef extra_spaces(t, r):\n    return re.sub(r" ", lambda m: "  " if r.random() < .15 else " ", t)\n\n\ndef homoglyph(t, r):\n    return _swap_chars(t, HOMOGLYPHS, .08, r)\n\n\ndef zero_width(t, r):\n    return re.sub(r" ", lambda m: ZERO_WIDTH + " " if r.random() < .3 else " ", t)\n\n\ndef drop_articles(t, r):\n    return re.sub(r"\\b(the|a|an|The|A|An)\\s", lambda m: "" if r.random() < .7 else m.group(0), t)\n\n\ndef strip_digits(t, r):\n    return re.sub(r"\\d", "", t)\n\n\ndef typos(t, r):\n    return _swap_words(t, TYPOS, .5, r)\n\n\ndef char_noise(t, r):\n    def sub(m):\n        w = m.group(0)\n        if len(w) < 5 or r.random() > .05:\n            return w\n        i = r.randrange(1, len(w) - 2)\n        return w[:i] + w[i + 1] + w[i] + w[i + 2:]\n    return re.sub(r"[A-Za-z]+", sub, t)\n\n\ndef spelling(t, r):\n    return _swap_words(t, SPELLING, 1., r)\n\n\ndef synonyms(t, r):\n    return _swap_words(t, SYNONYMS, .6, r)\n\n\ndef case_noise(t, r):\n    return re.sub(r"[A-Za-z]+", lambda m: m.group(0).swapcase() if r.random() < .05 else m.group(0), t)\n\n\nATTACKS = {f.__name__: f for f in (flatten_ws, drop_blank_lines, add_blank_lines, extra_spaces, homoglyph, zero_width,\n                                   drop_articles, strip_digits, typos, char_noise, spelling, synonyms, case_noise)}\n\n\ndef apply(name, texts, seed=0):\n    r = random.Random(seed)\n    return [ATTACKS[name](t, r) for t in texts]\n\n\ndef augment_random(texts, seed=0, k=2):\n    """apply k random edits to each text."""\n    r = random.Random(seed)\n    names = sorted(ATTACKS)\n    out = []\n    for t in texts:\n        for n in r.sample(names, k):\n            t = ATTACKS[n](t, r)\n        out.append(t)\n    return out\n', 'src/baseline.py': '"""the first notebook\'s lexical and catboost style recipe, kept as the reference to beat."""\nimport math\nimport re\nimport zlib\nfrom collections import Counter\n\nimport numpy as np\nfrom catboost import CatBoostClassifier\nfrom scipy.sparse import hstack\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\n\nSEED = 5785\nWORD = re.compile(r"[A-Za-z]+(?:[\'’][A-Za-z]+)?")\nFUNCTION_WORDS = sorted(ENGLISH_STOP_WORDS)\nPUNCT = ".,!?;:-—–\'’“”()[]/#*%&" + chr(34) + "\\n\\t"\n\n\ndef _dist(v):\n    return [np.mean(v), np.std(v), *np.quantile(v, [.1, .5, .9])] if v else [0.] * 5\n\n\ndef style_features(text):\n    words = WORD.findall(text)\n    lower = [w.lower() for w in words]\n    counts = Counter(lower)\n    chars = Counter(text)\n    nc, nw = max(1, len(text)), max(1, len(words))\n    sents = [p for p in re.split(r\'(?<=[.!?])\\s+|\\n+\', text) if p.strip()]\n    bg = Counter(zip(lower, lower[1:]))\n    b = text.encode()\n    return [\n        math.log1p(len(text)), math.log1p(len(words)), len(sents),\n        len(counts) / nw, len(counts) / math.sqrt(nw), sum(v == 1 for v in counts.values()) / nw,\n        max(counts.values(), default=0) / nw, len(bg) / nw, sum(v > 1 for v in bg.values()) / nw,\n        len(zlib.compress(b)) / max(1, len(b)), len(zlib.compress(" ".join(lower).encode())) / nw,\n        sum(c.isdigit() for c in text) / nc, sum(ord(c) > 127 for c in text) / nc,\n        sum(c.isupper() for c in text) / nc, sum(w.isupper() for w in words) / nw, sum(w.istitle() for w in words) / nw,\n        *_dist([len(w) for w in words]), *_dist([len(WORD.findall(s)) for s in sents]),\n        *_dist([len(s) for s in text.splitlines() if s.strip()]),\n        *[chars[c] / nc for c in PUNCT], *[counts[w] / nw for w in FUNCTION_WORDS],\n    ]\n\n\ndef style_matrix(texts):\n    return np.asarray([style_features(t) for t in texts], dtype=np.float32)\n\n\ndef fit_lexical(tr, y):\n    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80000, sublinear_tf=True,\n                           strip_accents="unicode", dtype=np.float32)\n    char = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=3, max_features=100000,\n                           sublinear_tf=True, dtype=np.float32)\n    x = hstack((word.fit_transform(tr), char.fit_transform(tr)), format="csr")\n    clf = LogisticRegression(C=8, solver="liblinear", max_iter=500, random_state=SEED).fit(x, y)\n    return lambda t: clf.decision_function(hstack((word.transform(t), char.transform(t)), format="csr"))\n\n\ndef fit_style(tr, y, iterations=850, depth=6, threads=8):\n    clf = CatBoostClassifier(iterations=iterations, depth=depth, learning_rate=.06, verbose=False, thread_count=threads,\n                             random_seed=SEED, allow_writing_files=False)\n    clf.fit(style_matrix(tr), y)\n    return lambda t: clf.predict(style_matrix(t), prediction_type="RawFormulaVal")\n', 'src/calibration.py': '"""Fit calibration on held-out training rows and keep inference independent of batch composition."""\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\n\n\nclass ScoreCalibration:\n    def fit(self, scores, labels):\n        scores = np.asarray(scores, dtype=float)\n        if scores.ndim != 2 or not np.isfinite(scores).all():\n            raise ValueError("Calibration requires a finite matrix of component scores.")\n        labels = np.asarray(labels)\n        if set(np.unique(labels)) != {0, 1}:\n            raise ValueError("Calibration needs both human and AI examples.")\n        self.scalers = []\n        self.models = []\n        for column in scores.T:\n            scaler = StandardScaler().fit(column[:, None])\n            model = LogisticRegression(C=1, random_state=3737).fit(scaler.transform(column[:, None]), labels)\n            if model.coef_[0, 0] <= 0:\n                raise ValueError("A component is inverted on the calibration set.")\n            self.scalers.append(scaler)\n            self.models.append(model)\n        return self\n\n    def transform(self, scores):\n        scores = np.asarray(scores, dtype=float)\n        if scores.ndim != 2 or scores.shape[1] != len(self.models) or not np.isfinite(scores).all():\n            raise ValueError("Prediction columns must match calibrated components.")\n        return np.column_stack([\n            model.predict_proba(scaler.transform(scores[:, index, None]))[:, 1]\n            for index, (scaler, model) in enumerate(zip(self.scalers, self.models))\n        ])\n\n\nclass RankCalibration:\n    """Held-out training quantiles; independent of the prediction batch."""\n    def fit(self, scores, labels=None):\n        from sklearn.preprocessing import QuantileTransformer\n        scores=np.asarray(scores,dtype=float)\n        if scores.ndim!=2 or not np.isfinite(scores).all():\n            raise ValueError(\'Rank calibration requires a finite component matrix.\')\n        self.mapping=QuantileTransformer(n_quantiles=min(800,len(scores)),subsample=None,\n                                         output_distribution=\'uniform\',random_state=3737).fit(scores)\n        return self\n\n    def transform(self,scores):\n        scores=np.asarray(scores,dtype=float)\n        if scores.ndim!=2 or not np.isfinite(scores).all():\n            raise ValueError(\'Prediction components must be a finite matrix.\')\n        return self.mapping.transform(scores)\n', 'src/encoder.py': '"""fine-tune a small pretrained encoder on cleaned text, with optional attack augmentation."""\nimport random\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\nfrom transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_cosine_schedule_with_warmup\n\nfrom .attacks import augment_random\nfrom .normalize import clean\n\nDEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef _seed(s):\n    random.seed(s); np.random.seed(s); torch.manual_seed(s)\n\n\ndef _collate(tok, max_len):\n    def f(batch):\n        text = [b[0] for b in batch]\n        enc = tok(text, padding=True, truncation=True, max_length=max_len, return_tensors="pt")\n        if len(batch[0]) > 1:\n            enc["labels"] = torch.tensor([b[1] for b in batch], dtype=torch.float32)\n            enc["w"] = torch.tensor([b[2] for b in batch], dtype=torch.float32)\n        return enc\n    return f\n\n\n@torch.no_grad()\ndef predict(model, tok, texts, max_len=256, batch=48):\n    model.eval()\n    order = np.argsort([len(t) for t in texts])[::-1]\n    cleaned_text = [clean(t) for t in texts]\n    out = np.zeros(len(texts))\n    for i in range(0, len(texts), batch):\n        ids = order[i:i + batch]\n        enc = tok([cleaned_text[j] for j in ids], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)\n        logit = model(**enc).logits.float().cpu().numpy()\n        out[ids] = logit[:, 1] - logit[:, 0]\n    return out\n\n\ndef train_encoder(texts, y, name="distilroberta-base", epochs=2, lr=2e-5, batch=16, max_len=256, aug_rate=0., weights=None,\n                  seed=3737, log_every=100, smoothing=0.05, dropout=0.):\n    _seed(seed)\n    tok = AutoTokenizer.from_pretrained(name, token=False)\n    # dropout off: with it on the model learns clean text means ai, see reports/FINDINGS.md section 8\n    cfg = AutoConfig.from_pretrained(name, token=False, num_labels=2, hidden_dropout_prob=dropout, attention_probs_dropout_prob=dropout,\n                                     classifier_dropout=dropout, attn_implementation="eager")\n    model = AutoModelForSequenceClassification.from_pretrained(name, config=cfg, use_safetensors=True, token=False).to(DEVICE)\n    w = np.ones(len(texts)) if weights is None else np.asarray(weights, dtype=float)\n    steps = epochs * (len(texts) // batch)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=.01)\n    sched = get_cosine_schedule_with_warmup(opt, int(.06 * steps), steps)\n    for ep in range(epochs):\n        t = list(texts)\n        if aug_rate:\n            r = random.Random(seed + ep)\n            pick = [i for i in range(len(t)) if r.random() < aug_rate]\n            for i, a in zip(pick, augment_random([t[i] for i in pick], seed=seed + ep)):\n                t[i] = a\n        data = [(clean(a), float(b), float(c)) for a, b, c in zip(t, y, w)]\n        loader = DataLoader(data, batch_size=batch, shuffle=True, collate_fn=_collate(tok, max_len),\n                            generator=torch.Generator().manual_seed(seed + ep))\n        model.train()\n        for i, enc in enumerate(loader):\n            yb, wb = enc.pop("labels").to(DEVICE), enc.pop("w").to(DEVICE)\n            enc = enc.to(DEVICE)\n            logit = model(**enc).logits\n            diff = logit[:, 1] - logit[:, 0]\n            target = yb * (1 - smoothing) + smoothing / 2\n            loss = (torch.nn.functional.binary_cross_entropy_with_logits(diff, target, reduction="none") * wb).mean()\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n            opt.step(); sched.step(); opt.zero_grad()\n            if i % log_every == 0:\n                print(f"ep {ep} step {i}/{len(loader)} loss {loss.item():.4f}", flush=True)\n    return model, tok\n', 'src/encoder_v2.py': '"""Full-precision DeBERTa fine-tuning with head-and-tail coverage and training-only validation."""\nimport gc\nimport json\nimport math\nfrom pathlib import Path\nimport random\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n\nfrom src.normalize import clean\n\n\nclass EncodedPassages(Dataset):\n    def __init__(self, texts, tokenizer, labels=None, max_length=256):\n        self.items = []\n        budget = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n        for index, text in enumerate(texts):\n            tokens = tokenizer.encode(clean(text), add_special_tokens=False, verbose=False)\n            if len(tokens) > budget:\n                tail = budget // 3\n                tokens = tokens[:budget-tail] + tokens[-tail:]\n            item = tokenizer.prepare_for_model(\n                tokens, add_special_tokens=True, return_attention_mask=True,\n                return_token_type_ids=False, verbose=False)\n            if labels is not None:\n                item["labels"] = int(labels[index])\n            self.items.append(item)\n\n    def __getitem__(self, index):\n        return self.items[index]\n\n    def __len__(self):\n        return len(self.items)\n\n\ndef device_name():\n    return "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"\n\n\ndef collator(tokenizer):\n    def collate(rows):\n        return tokenizer.pad(rows, padding=True, return_tensors="pt")\n    return collate\n\n\ndef fit_encoder(texts, labels, directory, seed=3737, epochs=2, model_name="microsoft/deberta-v3-base",\n                max_length=256, batch_size=8, learning_rate=1e-5, max_steps=None):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    device = device_name()\n    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False, token=False)\n    config = AutoConfig.from_pretrained(model_name, token=False)\n    config.num_labels = 2\n    for attribute in ("hidden_dropout_prob", "attention_probs_dropout_prob", "pooler_dropout", "cls_dropout"):\n        setattr(config, attribute, 0.)\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_name, config=config, token=False, use_safetensors=False).to(device)\n    # Keep the pretrained vocabulary geometry stable for words from unseen domains.\n    for parameter in model.base_model.embeddings.parameters():\n        parameter.requires_grad = False\n    dataset = EncodedPassages(texts, tokenizer, labels, max_length)\n    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collator(tokenizer),\n                        generator=torch.Generator().manual_seed(seed))\n    accumulation = 4\n    updates = epochs * math.ceil(len(loader) / accumulation)\n    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],\n                                  lr=learning_rate, weight_decay=.01)\n    schedule = get_linear_schedule_with_warmup(optimizer, int(updates * .08), updates)\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    import time\n    started = time.perf_counter()\n    seen_steps = 0\n    for epoch in range(epochs):\n        for step, batch in enumerate(loader):\n            batch = {name:value.to(device) for name,value in batch.items()}\n            # Correctly normalize the final partial accumulation group.\n            group_size = min(accumulation, len(loader) - (step // accumulation) * accumulation)\n            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits\n            loss = torch.nn.functional.cross_entropy(logits, batch["labels"], label_smoothing=.02)\n            if not bool(torch.isfinite(loss)):\n                raise RuntimeError("Non-finite encoder loss.")\n            (loss / group_size).backward()\n            if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n                optimizer.step()\n                schedule.step()\n                optimizer.zero_grad(set_to_none=True)\n            seen_steps += 1\n            if step % 50 == 0:\n                elapsed = time.perf_counter() - started\n                print(f"encoder epoch {epoch+1}/{epochs} step {step+1}/{len(loader)} "\n                      f"loss {loss.item():.4f} elapsed {elapsed/60:.1f} min", flush=True)\n            if max_steps is not None and seen_steps >= max_steps:\n                break\n        if max_steps is not None and seen_steps >= max_steps:\n            break\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    model.save_pretrained(directory)\n    tokenizer.save_pretrained(directory)\n    if max_steps is None:\n        (directory/"training_complete.json").write_text(json.dumps(\n            {"epochs":epochs,"rows":len(texts),"model":model_name,"seed":seed}\n        )+"\\n")\n    del model, optimizer\n    gc.collect()\n    if device == "cuda":\n        torch.cuda.empty_cache()\n\n\n@torch.inference_mode()\ndef predict_encoder(texts, directory, max_length=256, batch_size=16):\n    device = device_name()\n    tokenizer = AutoTokenizer.from_pretrained(directory, use_fast=False, token=False)\n    model = AutoModelForSequenceClassification.from_pretrained(directory).to(device).eval()\n    dataset = EncodedPassages(texts, tokenizer, max_length=max_length)\n    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator(tokenizer))\n    result = []\n    for batch in loader:\n        batch = {name:value.to(device) for name,value in batch.items()}\n        logits = model(**batch).logits.float().cpu().numpy()\n        result.extend(logits[:, 1] - logits[:, 0])\n    del model\n    gc.collect()\n    if device == "cuda":\n        torch.cuda.empty_cache()\n    return np.asarray(result)\n', 'src/lm_features.py': '"""Frozen small/large language-model signals and cross-model differences."""\nfrom pathlib import Path\nimport numpy as np\nfrom huggingface_hub import snapshot_download\nfrom .surprisal import surprisal_matrix\n\nMODELS=(\'gpt2\',\'openai-community/gpt2-large\')\nLARGE_REVISION=\'32b71b12589c2f8d625668d2335a01cac3249519\'\n\n\ndef resolve_model(model_id,offline_root=None):\n    if offline_root is not None:\n        path=Path(offline_root)/model_id.split(\'/\')[-1]\n        if not (path/\'config.json\').exists():raise ValueError(f\'Offline model is missing: {path}\')\n        return str(path)\n    if model_id==MODELS[1]:\n        return snapshot_download(model_id,revision=LARGE_REVISION,token=False,\n                                 allow_patterns=[\'*.json\',\'merges.txt\',\'vocab.json\',\'model.safetensors\'])\n    if model_id!=MODELS[0]:raise ValueError(\'Unexpected language model.\')\n    return model_id\n\n\ndef combine(small,large):\n    small=np.asarray(small);large=np.asarray(large)\n    if small.shape!=large.shape or small.ndim!=2 or small.shape[1]!=15:\n        raise ValueError(\'Expected matching N by 15 language-model feature matrices.\')\n    return np.c_[small,large,large-small]\n\n\ndef extract(texts,model_id,batch=8,offline_root=None,checkpoint=None):\n    return surprisal_matrix(texts,model_name=resolve_model(model_id,offline_root),batch=batch,\n                             log_every=100,checkpoint=checkpoint)\n', 'src/normalize.py': '"""undo evasion tricks before modeling so edits cannot flip the token stream."""\nimport re\nimport unicodedata\n\nCONFUSABLE = str.maketrans({"а": "a", "е": "e", "о": "o", "с": "c", "р": "p", "і": "i", "х": "x", "у": "y", "ѕ": "s",\n                            "ԁ": "d", "ɡ": "g", "һ": "h", "ј": "j", "ո": "n", "ν": "v", "ο": "o", "ı": "i",\n                            "’": "\'", "‘": "\'", "“": \'"\', "”": \'"\', "—": "-", "–": "-", "…": "...", "\\xa0": " "})\nZERO = re.compile("[\u200b-\u200f\u2060\ufeff\xad]")\nLATIN_MIX = re.compile(r"\\S+")\n\n\ndef _fix_token(tok):\n    has_latin = any("a" <= c.lower() <= "z" for c in tok)\n    return tok.translate(CONFUSABLE) if has_latin else tok\n\n\ndef clean(text):\n    """nfkc, strip zero width, fix mixed script tokens, flatten all whitespace."""\n    t = ZERO.sub("", unicodedata.normalize("NFKC", text))\n    t = LATIN_MIX.sub(lambda m: _fix_token(m.group(0)), t)\n    # A line wrap inside <br /> must not alter the encoder\'s token stream.\n    t = re.sub(r"\\s+", " ", t)\n    t = t.replace("<br />", " ").replace("<br/>", " ")\n    return re.sub(r"\\s+", " ", t).strip()\n', 'src/pipeline.py': '"""final model: reference lexical and style models, rank blended with a formatting-free surprisal model."""\nimport numpy as np\nfrom catboost import CatBoostClassifier\nfrom scipy.sparse import hstack\nfrom scipy.stats import rankdata\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\n\nfrom .baseline import style_matrix\n\nWEIGHTS = (.55, .25, .2)  # reference, surprisal, encoder. picked by experiment 21\n\n\nclass Blend:\n    def __init__(self, seed=3737, weights=WEIGHTS, threads=8, encoder_dir=None):\n        self.seed, self.weights, self.threads, self.encoder_dir = seed, weights, threads, encoder_dir\n\n    def _catboost(self, **kw):\n        return CatBoostClassifier(verbose=False, random_seed=self.seed, allow_writing_files=False, thread_count=self.threads, **kw)\n\n    def fit(self, texts, y, surprisal):\n        self.word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80000, sublinear_tf=True, strip_accents="unicode",\n                                    dtype=np.float32)\n        self.char = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=3, max_features=100000, sublinear_tf=True,\n                                    dtype=np.float32)\n        x = hstack((self.word.fit_transform(texts), self.char.fit_transform(texts)), format="csr")\n        self.lexical = LogisticRegression(C=8, solver="liblinear", max_iter=500, random_state=self.seed).fit(x, y)\n        self.style = self._catboost(iterations=850, depth=6, learning_rate=.06).fit(style_matrix(texts), y)\n        self.surprisal = self._catboost(iterations=500, depth=4, learning_rate=.04).fit(surprisal, y)\n        if self.weights[2] > 0:\n            from .encoder import train_encoder\n            model, tok = train_encoder(list(texts), y, seed=self.seed)\n            model.save_pretrained(self.encoder_dir)\n            tok.save_pretrained(self.encoder_dir)\n        return self\n\n    def _encoder_scores(self, texts):\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\n        from .encoder import DEVICE, predict\n        model = AutoModelForSequenceClassification.from_pretrained(self.encoder_dir, attn_implementation="eager").to(DEVICE)\n        return predict(model, AutoTokenizer.from_pretrained(self.encoder_dir), list(texts))\n\n    def parts(self, texts, surprisal):\n        x = hstack((self.word.transform(texts), self.char.transform(texts)), format="csr")\n        lex = self.lexical.decision_function(x)\n        sty = self.style.predict(style_matrix(texts), prediction_type="RawFormulaVal")\n        sur = self.surprisal.predict(surprisal, prediction_type="RawFormulaVal")\n        out = dict(reference=.35 * lex + .65 * sty, surprisal=sur)\n        if self.weights[2] > 0:\n            out["encoder"] = self._encoder_scores(texts)\n        return out\n\n    def score(self, texts, surprisal):\n        """values in (0, 1]. ranks within this batch, which is all auroc needs."""\n        p = self.parts(texts, surprisal)\n        names = ("reference", "surprisal", "encoder")\n        return sum(w * rankdata(p[n]) / len(texts) for w, n in zip(self.weights, names) if w > 0)\n', 'src/recipes.py': '"""Accepted inference recipes; base-model fitting is shared between them."""\nimport numpy as np\n\nBASELINE_WEIGHTS = np.array([.375, .225, .15, .25])\nCURRENT_WEIGHTS = np.array([.35, .234, .156, .26])\n\n\ndef recipe_weights(bundle, recipe="current"):\n    if recipe == "current":\n        weights = CURRENT_WEIGHTS\n    elif recipe == "baseline":\n        weights = BASELINE_WEIGHTS\n    elif recipe == "saved":\n        weights = bundle["ensemble"].weights\n    else:\n        raise ValueError(f"Unknown recipe: {recipe}")\n    weights = np.asarray(weights, dtype=float)\n    if weights.shape != (4,) or not np.isfinite(weights).all() or np.any(weights < 0) or not np.isclose(weights.sum(), 1):\n        raise ValueError("Expected four nonnegative, finite weights summing to one.")\n    return weights.copy()\n', 'src/robust_ensemble.py': '"""Four complementary methods with calibration fitted before inference."""\nimport numpy as np\nfrom .pipeline import Blend\nfrom .calibration import RankCalibration\n\nWEIGHTS=np.array([.375,.225,.15,.25])  # reference, dual-LM features, DistilRoBERTa, DeBERTa\n\n\ndef short_lines(text):\n    """Word-preserving stress view, identical to the validation transformation."""\n    import hashlib\n    import random\n    rng=random.Random(int.from_bytes(hashlib.sha256(text.encode()).digest()[:8],\'big\'))\n    words=text.split();lines=[];start=0\n    while start<len(words):\n        end=start+rng.randint(4,10);lines.append(\' \'.join(words[start:end]));start=end\n    return \'\\n\\n\'.join(\'\\n\'.join(lines[i:i+4]) for i in range(0,len(lines),4))\n\n\nclass RobustEnsemble:\n    def __init__(self,classical=None,calibration=None,weights=WEIGHTS):\n        self.classical=classical\n        self.calibration=calibration\n        self.weights=np.asarray(weights,dtype=float).copy()\n        if self.weights.shape!=(4,) or np.any(self.weights<0) or not np.isclose(self.weights.sum(),1):\n            raise ValueError(\'Expected four nonnegative weights summing to one.\')\n\n    @staticmethod\n    def fit_classical(texts,labels,features):\n        # Reuse the committed reference and LM recipe. No encoder is fitted here.\n        return Blend(seed=3737,weights=(.55,.25,0),threads=5).fit(texts,labels,features)\n\n    def components(self,texts,features,distil,deberta):\n        parts=self.classical.parts(texts,features)\n        return np.column_stack([parts[\'reference\'],parts[\'surprisal\'],distil,deberta])\n\n    def fit_calibration(self,texts,labels,features,distil,deberta):\n        views=(texts,[\' \'.join(s.split()) for s in texts],[short_lines(s) for s in texts])\n        raw=[self.components(view,features,distil,deberta) for view in views]\n        self.calibration=RankCalibration().fit(np.concatenate(raw))\n        return self\n\n    def score(self,texts,features,distil,deberta):\n        raw=self.components(texts,features,distil,deberta)\n        return np.clip(self.calibration.transform(raw)@self.weights,0,1)\n', 'src/surprisal.py': '"""token predictability under a pretrained lm, independent of this dataset\'s formatting.\n\ntext is whitespace-flattened first so paragraph breaks cannot leak in. the discrepancy feature\nfollows the analytic criterion from fast-detectgpt: observed log prob against its expectation\nunder the model\'s own predictive distribution.\n"""\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nNAMES = ["lp_mean", "lp_std", "lp_q10", "lp_q50", "lp_q90", "top1", "top5", "top10", "top100", "entropy", "discrepancy",\n         "lp_first", "lp_second", "lp_autocorr", "n_tokens"]\n\n\ndef flatten(t):\n    return re.sub(r"\\s+", " ", t).strip()\n\n\n@torch.no_grad()\ndef surprisal_matrix(texts, model_name="gpt2", max_len=384, batch=8, device=None, log_every=50, checkpoint=None):\n    device = device or ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")\n    tok = AutoTokenizer.from_pretrained(model_name, token=False)\n    tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    model = AutoModelForCausalLM.from_pretrained(model_name, use_safetensors=True, token=False).to(device).eval()\n    out = np.zeros((len(texts), len(NAMES)), dtype=np.float32)\n    order = np.argsort([len(t) for t in texts])\n    first = 0\n    if checkpoint and Path(checkpoint).exists():\n        saved = np.load(checkpoint)\n        out, first = saved["out"], int(saved["next"])\n        print("resuming at", first, flush=True)\n    for b, start in enumerate(range(first, len(texts), batch)):\n        ids = order[start:start + batch]\n        enc = tok([flatten(texts[i]) for i in ids], return_tensors="pt", padding=True, truncation=True,\n                  max_length=max_len).to(device)\n        logp = torch.log_softmax(model(**enc).logits[:, :-1], -1)\n        tgt, mask = enc.input_ids[:, 1:], enc.attention_mask[:, 1:].bool()\n        lp = logp.gather(-1, tgt[..., None])[..., 0]\n        p = logp.exp()\n        ent = -(p * logp).sum(-1)\n        var = (p * logp.pow(2)).sum(-1) - ent.pow(2)\n        rank = (logp > lp[..., None]).sum(-1)\n        lp, ent, var, rank, mask = [x.float().cpu().numpy() if x is not mask else x.cpu().numpy()\n                                    for x in (lp, ent, var, rank, mask)]\n        for row, i in enumerate(ids):\n            m = mask[row]\n            if m.sum() < 3:\n                continue\n            a, r = lp[row][m], rank[row][m]\n            h = len(a) // 2\n            out[i] = [a.mean(), a.std(), *np.quantile(a, [.1, .5, .9]), (r < 1).mean(), (r < 5).mean(), (r < 10).mean(),\n                      (r < 100).mean(), ent[row][m].mean(),\n                      (a.sum() + ent[row][m].sum()) / np.sqrt(var[row][m].sum() + 1e-6),\n                      a[:h].mean(), a[h:].mean(), np.corrcoef(a[:-1], a[1:])[0, 1], len(a)]\n        if b % log_every == 0:\n            print(f"{start}/{len(texts)}", flush=True)\n            torch.mps.empty_cache() if device == "mps" else None\n            if checkpoint:\n                np.savez(checkpoint, out=out, next=start + batch)\n    return np.nan_to_num(out)\n', 'train_robust.py': '"""Train the fixed four-method robustness candidate using supplied data only."""\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.model_selection import train_test_split\nfrom src.encoder import train_encoder,predict as legacy_predict,DEVICE\nfrom src.encoder_v2 import fit_encoder,predict_encoder\nfrom src.robust_ensemble import RobustEnsemble\nfrom src.lm_features import MODELS,LARGE_REVISION,extract,combine\nfrom src.recipes import BASELINE_WEIGHTS,CURRENT_WEIGHTS\n\nTRAINING_SHA256=\'325e345faf88fb1c4203278098c282e40319a87b4bf17e094d2f235f51d0c07e\'\n\n\ndef atomic_array(path,**arrays):\n    temporary=path.with_suffix(\'.tmp\')\n    with temporary.open(\'wb\') as handle:np.savez(handle,**arrays)\n    temporary.replace(path)\n\n\ndef partitions(labels):\n    dev,audit=train_test_split(np.arange(len(labels)),test_size=2000,stratify=labels,random_state=3737)\n    fit,meta=train_test_split(dev,test_size=1600,stratify=labels[dev],random_state=9407)\n    cal,selection=train_test_split(meta,test_size=800,stratify=labels[meta],random_state=9413)\n    return fit,cal,selection,audit\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--small-features\',type=Path);p.add_argument(\'--large-features\',type=Path)\n    p.add_argument(\'--validation\',action=\'store_true\',help=\'Fit 6400 rows and reserve the frozen 2000-row audit instead of final refit\')\n    p.add_argument(\'--recipe\',choices=(\'current\',\'baseline\'),default=None,\n                   help=\'Defaults to current for final fitting, baseline for reproducing the original validation audit\')\n    a=p.parse_args();started=time.perf_counter()\n    recipe=a.recipe or (\'baseline\' if a.validation else \'current\')\n    inference_weights=CURRENT_WEIGHTS if recipe==\'current\' else BASELINE_WEIGHTS\n    tr=pd.read_csv(a.data/\'train.csv\',keep_default_na=False)\n    digest=hashlib.sha256(tr[[\'id\',\'text\',\'label\']].to_csv(index=False).encode()).hexdigest()\n    if digest!=TRAINING_SHA256:raise ValueError(\'Use the original competition train.csv to reproduce this validated recipe.\')\n    t=tr.text.to_numpy();y=tr.label.to_numpy();base_fit,cal,_,_=partitions(y)\n    fit=base_fit if a.validation else np.setdiff1d(np.arange(len(tr)),cal)\n    stage=\'validation\' if a.validation else \'final\'\n    source=Path(__file__).parent\n    signature=hashlib.sha256(Path(__file__).read_bytes()+b\'\'.join(path.read_bytes() for path in sorted((source/\'src\').glob(\'*.py\')))).hexdigest()\n    config={\'training_sha256\':digest,\'source_sha256\':signature,\'weights\':inference_weights.tolist(),\'inference_recipe\':recipe,\n            \'fit_rows\':len(fit),\'calibration_rows\':len(cal),\'encoder_epochs\':2,\'recipe\':\'four-method-robustness-v1\',\'stage\':stage}\n    a.output.mkdir(parents=True,exist_ok=True);manifest=a.output/\'training_config.json\'\n    if manifest.exists() and json.loads(manifest.read_text())!=config:\n        raise ValueError(\'This output folder belongs to another recipe. Choose a fresh output folder.\')\n    manifest.write_text(json.dumps(config,indent=2)+\'\\n\')\n    atomic_array(a.output/\'partitions.npz\',fit=fit,calibration=cal)\n    matrices=[]\n    for model_id,supplied,name in zip(MODELS,(a.small_features,a.large_features),(\'small\',\'large\')):\n        feature_path=supplied or a.output/f\'{name}_features.npz\'\n        if feature_path.exists():\n            with np.load(feature_path) as stored:\n                if str(stored[\'digest\'])!=digest:raise ValueError(\'Feature cache dataset mismatch.\')\n                values=stored[\'features\']\n                if \'model\' in stored and str(stored[\'model\'])!=model_id:raise ValueError(\'Feature model mismatch.\')\n                if name==\'large\' and (\'revision\' not in stored or str(stored[\'revision\'])!=LARGE_REVISION):\n                    raise ValueError(\'Large-model feature revision mismatch.\')\n        else:\n            if supplied:raise FileNotFoundError(feature_path)\n            values=extract(t,model_id,batch=8,checkpoint=a.output/f\'{name}_partial.npz\')\n            metadata={\'model\':model_id,\'revision\':LARGE_REVISION if name==\'large\' else \'gpt2\'}\n            atomic_array(feature_path,features=values,digest=digest,**metadata)\n        if values.shape!=(len(tr),15) or not np.isfinite(values).all():raise ValueError(\'Invalid language-model feature cache.\')\n        matrices.append(values)\n        print(f\'{model_id} features ready\',flush=True)\n    features=combine(*matrices)\n    classical_path=a.output/\'classical.joblib\'\n    if classical_path.exists():classical=joblib.load(classical_path)\n    else:\n        classical=RobustEnsemble.fit_classical(t[fit],y[fit],features[fit]);joblib.dump(classical,classical_path)\n    print(\'Reference and language-feature classifiers ready\',flush=True)\n    deb_dir=a.output/\'deberta\'\n    if not (deb_dir/\'training_complete.json\').exists():fit_encoder(t[fit],y[fit],deb_dir)\n    distil_dir=a.output/\'distilroberta\'\n    if not (distil_dir/\'training_complete.json\').exists():\n        model,tok=train_encoder(list(t[fit]),y[fit],log_every=100)\n        model.save_pretrained(distil_dir);tok.save_pretrained(distil_dir)\n        (distil_dir/\'training_complete.json\').write_text(json.dumps({\'epochs\':2,\'rows\':len(fit),\'seed\':3737})+\'\\n\')\n        del model,tok;gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n    # Calibration rows are excluded from every supervised component fit.\n    deberta=predict_encoder(t[cal],deb_dir)\n    from transformers import AutoModelForSequenceClassification,AutoTokenizer\n    model=AutoModelForSequenceClassification.from_pretrained(distil_dir,token=False,attn_implementation=\'eager\').to(DEVICE)\n    distil=legacy_predict(model,AutoTokenizer.from_pretrained(distil_dir,token=False),list(t[cal]))\n    del model;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    ensemble=RobustEnsemble(classical,weights=inference_weights).fit_calibration(t[cal],y[cal],features[cal],distil,deberta)\n    joblib.dump({\'ensemble\':ensemble,\'deberta_dir\':deb_dir.name,\'distil_dir\':distil_dir.name,\n                 \'training_sha256\':digest,\'stage\':stage,\'fit_rows\':len(fit),\'calibration_rows\':len(cal),\n                 \'lm_models\':list(MODELS),\'lm_feature_layout\':\'small,large,large-minus-small\',\'inference_recipe\':recipe},\n                a.output/\'model.joblib\')\n    config[\'elapsed_minutes\']=(time.perf_counter()-started)/60\n    config[\'python\']=sys.version\n    config[\'torch\']=torch.__version__\n    config[\'device\']=torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE\n    if torch.cuda.is_available():config[\'peak_gpu_gib\']=torch.cuda.max_memory_allocated()/2**30\n    (a.output/\'training_complete.json\').write_text(json.dumps(config,indent=2)+\'\\n\')\n    print(json.dumps(config,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'validate_content_robust.py': '"""Fixed four-method recipe tested on topics excluded from supervised fitting.\n\nUses only the original 6400-row fitting partition. Calibration is drawn from the\nremaining topics. Weights are locked; held-out topic labels never select them.\n"""\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nfrom sklearn.cluster import KMeans\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import normalize\nfrom src.calibration import RankCalibration\nfrom src.robust_ensemble import RobustEnsemble\nfrom src.encoder import DEVICE,predict as predict_legacy\nfrom transformers import AutoModelForSequenceClassification,AutoTokenizer\n\nfrom src.calibration import ScoreCalibration\nfrom src.encoder import train_encoder\nfrom src.encoder_v2 import fit_encoder,predict_encoder\nfrom src.robust_ensemble import short_lines\nfrom train_robust import atomic_array\n\ndef legacy_scores(texts,directory):\n    model=AutoModelForSequenceClassification.from_pretrained(directory,token=False,attn_implementation=\'eager\').to(DEVICE)\n    result=predict_legacy(model,AutoTokenizer.from_pretrained(directory,token=False),list(texts))\n    del model;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return result\n\n\ndef rank_features(values):\n    return np.column_stack([np.rint(2*rankdata(v)).astype(np.int64) for v in values.T])\n\n\ndef control_score(values):\n    return (rank_features(values)@np.array([55,25,20,0],dtype=np.int64))/(200*len(values))\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',type=Path,required=True)\n    p.add_argument(\'--features\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--recipe\',type=Path,required=True);p.add_argument(\'--large-features\',type=Path)\n    a=p.parse_args();a.output.mkdir(parents=True,exist_ok=True)\n    tr=pd.read_csv(a.data/\'train.csv\',keep_default_na=False)\n    t=tr.text.to_numpy();y=tr.label.to_numpy()\n    digest=hashlib.sha256(tr[[\'id\',\'text\',\'label\']].to_csv(index=False).encode()).hexdigest()\n    cached=np.load(a.features)\n    if str(cached[\'digest\'])!=digest:raise ValueError(\'Feature cache dataset mismatch.\')\n    features=cached[\'features\']\n    recipe=json.loads(a.recipe.read_text());weights=np.asarray(recipe[\'weights\']);mode=recipe[\'score_mode\']\n    if mode!=\'quantile\' or recipe[\'lm_mode\']!=\'both\':raise ValueError(\'Use the frozen validated dual-LM quantile recipe.\')\n    if recipe[\'training_sha256\']!=digest:raise ValueError(\'Recipe dataset mismatch.\')\n    candidate_features=features\n    if recipe[\'lm_mode\']!=\'small\':\n        if a.large_features is None:raise ValueError(\'This recipe requires --large-features.\')\n        large=np.load(a.large_features)\n        if str(large[\'digest\'])!=digest:raise ValueError(\'Larger LM dataset mismatch.\')\n        candidate_features=large[\'features\'] if recipe[\'lm_mode\']==\'large\' else np.c_[features,large[\'features\'],large[\'features\']-features]\n    dev,_=train_test_split(np.arange(len(tr)),test_size=2000,stratify=y,random_state=3737)\n    original_fit,_=train_test_split(dev,test_size=1600,stratify=y[dev],random_state=9407)\n    vectorizer=TfidfVectorizer(stop_words=\'english\',min_df=5,max_df=.4,max_features=40000,sublinear_tf=True)\n    x=vectorizer.fit_transform(t[original_fit])\n    z=normalize(TruncatedSVD(60,random_state=3737).fit_transform(x))\n    groups=KMeans(6,n_init=5,random_state=3737).fit_predict(z)\n    vocabulary=np.array(vectorizer.get_feature_names_out())\n    top={g:vocabulary[np.argsort(np.asarray(x[groups==g].mean(0)).ravel())[-15:]].tolist() for g in range(6)}\n    vocabulary_targets={\'news\':{\'government\',\'company\',\'mr\',\'uk\',\'market\',\'year\',\'said\'}}\n    chosen={name:max(range(6),key=lambda g:len(set(top[g])&words)) for name,words in vocabulary_targets.items()}\n    if len(set(chosen.values()))!=len(chosen):raise ValueError(\'Content groups could not be distinguished.\')\n    source=Path(__file__).parent\n    fingerprint=hashlib.sha256(Path(__file__).read_bytes()+b\'\'.join(path.read_bytes() for path in sorted((source/\'src\').glob(\'*.py\')))).hexdigest()\n    config={\'dataset_sha256\':digest,\'source_sha256\':fingerprint,\'weights\':weights.tolist(),\'recipe\':recipe,\'topics\':chosen,\'topic_terms\':top}\n    config=json.loads(json.dumps(config))\n    manifest=a.output/\'config.json\'\n    if manifest.exists() and json.loads(manifest.read_text())!=config:raise ValueError(\'Choose a fresh output directory for another experiment.\')\n    manifest.write_text(json.dumps(config,indent=2)+\'\\n\')\n    reports=[];all_labels=[];pooled_new={name:[] for name in (\'original\',\'flat\',\'short_lines\')};pooled_old={name:[] for name in pooled_new}\n    for topic,group in chosen.items():\n        held=original_fit[groups==group];rest=original_fit[groups!=group]\n        fit,cal=train_test_split(rest,test_size=800,stratify=y[rest],random_state=9407)\n        directory=a.output/topic;directory.mkdir(exist_ok=True)\n        print(f\'Content holdout {topic}: fit {len(fit)}, calibration {len(cal)}, held out {len(held)}\',flush=True)\n        np.savez(directory/\'partitions.npz\',fit=fit,calibration=cal,held=held)\n        classical_path=directory/\'classical.joblib\'\n        if classical_path.exists():models=joblib.load(classical_path)\n        else:\n            reference=RobustEnsemble.fit_classical(t[fit],y[fit],features[fit])\n            lm=reference.surprisal\n            candidate_lm=lm if recipe[\'lm_mode\']==\'small\' else CatBoostClassifier(iterations=500,depth=4,learning_rate=.04,random_seed=3737,thread_count=5,verbose=False,allow_writing_files=False).fit(candidate_features[fit],y[fit])\n            models={\'reference\':reference,\'lm\':lm,\'candidate_lm\':candidate_lm};joblib.dump(models,classical_path)\n        reference=models[\'reference\'];lm=models[\'lm\']\n        ordered=np.r_[cal,held];scores_path=directory/\'encoder_scores.npz\'\n        if scores_path.exists():\n            saved=np.load(scores_path);deberta=saved[\'deberta\'];distil=saved[\'distil\']\n            if not np.array_equal(saved[\'ordered\'],ordered):raise ValueError(\'Encoder score split mismatch.\')\n        else:\n            deb_dir=directory/\'deberta\'\n            if not (deb_dir/\'training_complete.json\').exists():fit_encoder(t[fit],y[fit],deb_dir)\n            deberta=predict_encoder(t[ordered],deb_dir)\n            distil_dir=directory/\'distilroberta\'\n            if not (distil_dir/\'training_complete.json\').exists():\n                model,tok=train_encoder(list(t[fit]),y[fit],log_every=100)\n                model.save_pretrained(distil_dir);tok.save_pretrained(distil_dir)\n                (distil_dir/\'training_complete.json\').write_text(json.dumps({\'rows\':len(fit),\'epochs\':2})+\'\\n\')\n                del model,tok;gc.collect()\n                if torch.cuda.is_available():torch.cuda.empty_cache()\n            distil=legacy_scores(t[ordered],distil_dir)\n            atomic_array(scores_path,ordered=ordered,deberta=deberta,distil=distil)\n        lm_score=lm.predict(features[ordered],prediction_type=\'RawFormulaVal\')\n        candidate_lm_score=models[\'candidate_lm\'].predict(candidate_features[ordered],prediction_type=\'RawFormulaVal\')\n        raw={}\n        for name,view in [(\'original\',t[ordered]),(\'flat\',[\' \'.join(s.split()) for s in t[ordered]]),\n                          (\'short_lines\',[short_lines(s) for s in t[ordered]])]:\n            raw[name]=np.c_[reference.parts(view,features[ordered])[\'reference\'],candidate_lm_score,distil,deberta]\n        calibration=RankCalibration().fit(np.concatenate([v[:len(cal)] for v in raw.values()]),np.tile(y[cal],len(raw)))\n        report={\'topic\':topic,\'terms\':top[group],\'fit_rows\':len(fit),\'calibration_rows\':len(cal),\n                \'held_rows\':len(held),\'held_human\':int(sum(y[held]==0)),\'held_ai\':int(sum(y[held]==1)),\n                \'candidate\':{},\'control\':{}}\n        arrays={\'ids\':tr.id.to_numpy()[held],\'labels\':y[held]}\n        all_labels.append(y[held])\n        for name,values in raw.items():\n            test=values[len(cal):]\n            new=calibration.transform(test)@weights\n            control=test.copy();control[:,1]=lm_score[len(cal):]\n            old=control_score(control)\n            report[\'candidate\'][name]=float(roc_auc_score(y[held],new));report[\'control\'][name]=float(roc_auc_score(y[held],old))\n            pooled_new[name].append(new);pooled_old[name].append(old)\n            arrays[\'candidate_\'+name]=new;arrays[\'control_\'+name]=old\n        atomic_array(directory/\'scores.npz\',**arrays)\n        (directory/\'report.json\').write_text(json.dumps(report,indent=2)+\'\\n\');reports.append(report)\n        print(json.dumps(report,indent=2),flush=True)\n    labels=np.concatenate(all_labels)\n    result={\'weights_locked_before_testing\':weights.tolist(),\'score_mode\':mode,\'lm_mode\':recipe[\'lm_mode\'],\'topics\':reports,\n            \'pooled_candidate\':{name:float(roc_auc_score(labels,np.concatenate(values))) for name,values in pooled_new.items()},\n            \'pooled_control\':{name:float(roc_auc_score(labels,np.concatenate(values))) for name,values in pooled_old.items()},\n            \'scope\':\'Additional content group with frozen recipe; from the original fitting partition only, not a human-poetry benchmark.\'}\n    (a.output/\'domain_ensemble.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n    print(json.dumps(result,indent=2),flush=True)\n\nif __name__==\'__main__\':main()\n', 'validate_robust.py': '"""Reproduce the fixed global audit; run train_robust.py --validation first."""\nimport argparse,json\nfrom pathlib import Path\nimport joblib,numpy as np,pandas as pd\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\nfrom src.lm_features import MODELS,extract,combine\nfrom src.robust_ensemble import short_lines\nfrom src.encoder_v2 import predict_encoder\nfrom predict_robust import DEVICE,legacy_predict,AutoModelForSequenceClassification,AutoTokenizer\nfrom train_robust import partitions,atomic_array\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--model\',type=Path,required=True)\n    a=p.parse_args();bundle=joblib.load(a.model/\'model.joblib\')\n    if bundle[\'stage\']!=\'validation\':raise ValueError(\'Audit requires a validation fit, not a final fit that has seen audit labels.\')\n    frame=pd.read_csv(a.data/\'train.csv\',keep_default_na=False);texts=frame.text.to_numpy();y=frame.label.to_numpy()\n    fit,_,_,audit=partitions(y)\n    matrices=[]\n    for model_id,name in zip(MODELS,(\'small\',\'large\')):\n        path=a.model/f\'{name}_features.npz\'\n        matrices.append(np.load(path)[\'features\'] if path.exists() else extract(texts,model_id))\n    features=combine(*matrices)\n    small=CatBoostClassifier(iterations=500,depth=4,learning_rate=.04,random_seed=3737,thread_count=5,\n                            verbose=False,allow_writing_files=False).fit(matrices[0][fit],y[fit])\n    lm_control=small.predict(matrices[0][audit],prediction_type=\'RawFormulaVal\')\n    deb=predict_encoder(texts[audit],a.model/bundle[\'deberta_dir\'])\n    distil_dir=a.model/bundle[\'distil_dir\']\n    model=AutoModelForSequenceClassification.from_pretrained(distil_dir,token=False,attn_implementation=\'eager\').to(DEVICE)\n    distil=legacy_predict(model,AutoTokenizer.from_pretrained(distil_dir,token=False),list(texts[audit]))\n    report={};arrays={\'ids\':frame.id.to_numpy()[audit],\'labels\':y[audit]}\n    for name,transform in [(\'original\',lambda s:s),(\'flat\',lambda s:\' \'.join(s.split())),(\'short_lines\',short_lines)]:\n        view=[transform(s) for s in texts[audit]]\n        raw=bundle[\'ensemble\'].components(view,features[audit],distil,deb)\n        candidate=bundle[\'ensemble\'].score(view,features[audit],distil,deb)\n        # Canonical committed three-way blend, on exactly the same fit rows.\n        control=sum(w*np.rint(2*rankdata(column,method=\'average\')).astype(np.int64) for w,column in zip((55,25,20),(raw[:,0],lm_control,distil)))/(200*len(audit))\n        report[name]={\'candidate_auc\':roc_auc_score(y[audit],candidate),\'control_auc\':roc_auc_score(y[audit],control)}\n        arrays[\'candidate_\'+name]=candidate;arrays[\'control_\'+name]=control\n    atomic_array(a.model/\'audit_scores.npz\',**arrays)\n    (a.model/\'audit_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\');print(json.dumps(report,indent=2))\n\nif __name__==\'__main__\':main()\n'}
for relative,source in FILES.items():
    target=PROJECT/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements-robust.txt')],check=True)


In [ ]:
import torch, zipfile
from google.colab import files
if not torch.cuda.is_available():raise RuntimeError('Select a T4 GPU runtime before continuing.')
print('GPU:',torch.cuda.get_device_name(0))
DATA=PROJECT/'data';DATA.mkdir(exist_ok=True)
required=('train.csv','test.csv','sample_submission.csv')
if not all((DATA/name).exists() for name in required):
    uploaded=files.upload()
    archives=[Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(archives)!=1:raise ValueError('Upload the single original competition ZIP.')
    with zipfile.ZipFile(archives[0]) as archive:
        for name in required:
            matches=[entry for entry in archive.namelist() if Path(entry).name==name and not entry.startswith('__MACOSX/')]
            if len(matches)!=1:raise ValueError(f'Expected exactly one {name} in the ZIP.')
            (DATA/name).write_bytes(archive.read(matches[0]))


In [ ]:
def run(script,*arguments):
    process=subprocess.Popen([sys.executable,'-u',str(PROJECT/script),*map(str,arguments)],
        cwd=PROJECT,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:print(line,end='',flush=True)
    if process.wait():raise RuntimeError(f'{script} failed; inspect the output above.')
MODEL=PROJECT/'final_model'
run('train_robust.py','--data',DATA,'--output',MODEL)


In [ ]:
run('check_robust.py','--model',MODEL,'--input',DATA/'test.csv','--recipe','current')


In [ ]:
PREDICTIONS=PROJECT/'submission_four_methods.csv'
run('predict_robust.py','--model',MODEL,'--input',DATA/'test.csv','--output',PREDICTIONS,'--recipe','current')


In [ ]:
import pandas as pd,numpy as np,shutil
pred=pd.read_csv(PREDICTIONS);sample=pd.read_csv(DATA/'sample_submission.csv')
assert list(pred.columns)==['id','label']
assert pred.id.equals(sample.id) and not pred.id.duplicated().any()
assert np.isfinite(pred.label).all() and pred.label.between(0,1).all()
print('Validated prediction rows:',len(pred))
# Include fitted classifier weights, source, and provenance. Exclude competition data and training feature caches.
BUNDLE=PROJECT/'four_method_bundle.zip'
run('package_robust.py','--model',MODEL,'--output',BUNDLE)
print('Save both downloads before disconnecting Colab.')
files.download(str(PREDICTIONS))
files.download(str(BUNDLE))


## Score the professor’s new CSV later

Unzip `four_method_bundle.zip`, install `requirements-robust.txt`, then run:

```bash
USE_TF=0 USE_FLAX=0 python predict_robust.py --model final_model --input new_test.csv --output predictions.csv --recipe current
```

The new CSV needs `id,text`; no labels are required. Public GPT-2 weights download automatically. To work offline, supply `--offline-root` with `gpt2/` and `gpt2-large/` folders containing the original pretrained models/tokenizers. Fine-tuned neural classifiers are already in the bundle. Calibration and weights are fixed, so scoring a different set of rows does not refit the detector.

The bundle is approximately 0.8 GB before adding pretrained GPT-2 weights to an optional offline deployment. Its contents belong in shared storage, not GitHub source commits.
